# Quick start: Breaking the Blend on a mock galaxy

This notebook runs the **whole method in a few minutes without any JWST data**, on synthetic MIRI/MRS
cubes of a galaxy with a rotating star-forming ring and an AGN-driven bicone. Because we know the true
answer, every step can be checked. It is the hands-on introduction; `01_NGC5728.ipynb` applies the same
code to real data.

The five steps:
1. measure each line (flux, velocities, dispersions) and get the typical **dispersion of each component**;
2. fit **velocity models** of the disc and of the outflow;
3. **constrained decomposition**: in each spaxel, fix the velocities and widths of the two Gaussians and fit
   only their amplitudes -> disc fraction per line;
4. **stack** all lines into one AGN-fraction map;
5. extract **AGN- and star-formation-dominated spectra**.

In [ ]:
# ---- figure output: "png", "pdf", "both" or "none"  (saved in results/MockGalaxy/figures/) ----
SAVE_FIGURES = "both"
FIGURE_DPI   = 200

%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
try:
    import breaking_the_blend as btb
except ModuleNotFoundError:   # not installed (pip install -e .): use the copy in this repository
    import sys, os
    sys.path.insert(0, os.path.abspath(".."))
    import breaking_the_blend as btb
from breaking_the_blend import plotting as bplt     # also sets the publication plotting style

proj = btb.Project("MockGalaxy", results_dir="results", save_figures=SAVE_FIGURES, figure_dpi=FIGURE_DPI)

## How the mock galaxy is made

`btb.mock.make_mock_cubes` builds one cube per instrument band:

1. **Bands** - the 12 real MIRI/MRS sub-bands (`btb.MRS_BANDS`: observed wavelength range, pixel size,
   resolving power). The wavelength axis is made in the **observed frame** and divided by $(1+z)$, exactly as
   `load_cube` does for real data, so each cube covers slightly bluer rest-frame wavelengths.
2. **Geometry** - every band covers the same 7" field on its own pixel grid (0.13"-0.35"), AGN near the centre.
3. **Two gas components**:
   * a star-forming **ring** (radius 500 pc) in an inclined thin disc rotating with a tanh rotation curve;
   * a hollow **bicone** whose gas moves radially (rise to 600 km/s at 500 pc, then decline), plus a compact nucleus.

   Velocities come from the same `models` used to fit the data (`btb.mock.TRUE_PARAMS`).
4. **Lines** - every catalogue line falling in a band is a disc Gaussian ($\sigma$ = 70 km/s) + an outflow Gaussian
   ($\sigma$ = 200 km/s). The **disc share decreases with ionisation potential (IP)**, as observed in AGN.
5. **Continuum and noise** - a rising AGN point source (PSF growing with wavelength), a faint disc continuum,
   and Gaussian noise.

`truth` holds the true parameters and the true disc-fraction map of every line.

In [ ]:
Z = 0.0093
MOCK_BANDS = btb.MRS_BANDS      # {band: (lambda_min, lambda_max [observed micron], pixel ["], resolving power)}

print(f"{'band':>11s} {'observed [um]':>15s} {'rest frame [um]':>17s} {'pixel':>7s}")
for band, (lo, hi, pix, R) in MOCK_BANDS.items():
    print(f"{band:>11s} {lo:6.2f} - {hi:5.2f}   {lo/(1+Z):6.2f} - {hi/(1+Z):6.2f}   {pix:.2f}\"")

cubes, truth = btb.mock.make_mock_cubes(MOCK_BANDS, z=Z)
btb.set_agn_positions(cubes)
fig = bplt.plot_cubes_overview(cubes, ncols=6);  proj.savefig(fig, "cubes_overview")

## Choose lines

`btb.get_lines` takes lines from the catalogue (ordered by IP). Any setting can be changed with `.update(...)`.
Always check for **neighbouring lines** inside a line's window: [Cl II] 14.368 $\mu$m sits ~1000 km/s from
[Ne V] 14.32 $\mu$m (try removing the [Ne V] line below and look at its velocity map!).

In [ ]:
LINES = ["FeII", "ArII", "NeII", "SIII", "ArIII", "SIV", "NeIII", "OIV", "NeV"]
lines = {n: l.update(mask_percentile=40) for n, l in btb.get_lines(LINES).items()}
lines["NeV"] = lines["NeV"].update(line_region=(14.295, 14.345), fit_window=(14.265, 14.350))

lines = btb.assign_cubes(lines, cubes)       # picks the band covering each line

## Step 1 - measure the lines

In [ ]:
line_maps = proj.checkpoint("step1_line_maps", btb.fit_lines, cubes, lines, settings=dict(lines=lines))
fig = bplt.plot_spaxel_fits(line_maps["NeIII"], n=6);  proj.savefig(fig, "spaxel_fits_NeIII")

### Flux and single-Gaussian velocity maps

In [ ]:
fig = bplt.plot_line_maps(line_maps, "flux");      proj.savefig(fig, "flux_maps")
fig = bplt.plot_line_maps(line_maps, "velocity");  proj.savefig(fig, "velocity_maps")

The velocity maps change from disc rotation (low IP) to the bicone (high IP). Velocity dispersion of the single-Gaussian fits:

In [ ]:
fig = bplt.plot_line_maps(line_maps, "sigma");  proj.savefig(fig, "dispersion_maps")

### Double-Gaussian maps

Narrow (left) and broad (right) component of the double-Gaussian fit for each line - velocities, then dispersions.
The narrow component follows the disc, the broad one the outflow.

In [ ]:
fig = bplt.plot_line_quantities(line_maps, ("v_n", "v_b"), lines_per_row=3)
proj.savefig(fig, "double_gaussian_velocity_maps")

In [ ]:
fig = bplt.plot_line_quantities(line_maps, ("sigma_n", "sigma_b"), lines_per_row=3)
proj.savefig(fig, "double_gaussian_dispersion_maps")

### Moment maps

Model-free moments of the continuum-subtracted line: 0 (integrated flux, normalised to the peak of each line),
1 (flux-weighted velocity) and 2 (flux-weighted dispersion).

In [ ]:
fig = bplt.plot_line_quantities(line_maps, ("mom0", "mom1", "mom2"), lines_per_row=3)
proj.savefig(fig, "moment_maps")

### Dispersion of each component (true values: 70 and 200 km/s)

In [ ]:
sigmas = btb.dispersion_medians(line_maps)
fig = bplt.plot_dispersion_histograms(line_maps, sigmas);  proj.savefig(fig, "dispersion_histograms")

**Reading the histograms.** In most lines the two distributions separate cleanly. Two features are worth explaining:

* **Broad-component values at the narrow dispersion** (green bar at ~70 km s$^{-1}$). In spaxels that contain only
  disc emission - the parts of the ring outside the bicone - the line profile is a single Gaussian. A double-Gaussian
  fit to a single-component profile is degenerate: both Gaussians end up describing the same profile, sharing its
  flux with (nearly) the same $\sigma$, so after ordering by width the "broad" one also sits at the narrow value. These
  spaxels are not wrong fits; they simply carry no information about the outflow width. The same happens for real data
  wherever one component dominates.
* **A low-$\sigma$ tail of the narrow component** (< 40 km s$^{-1}$): in fainter spaxels the narrow Gaussian can latch onto a
  noise spike.

Because the adopted values are medians over hundreds of spaxels, neither feature changes them much. They can be
excluded by requiring the broad component to be clearly broader, e.g. `min_ratio=1.3` (next cell). [Ne V] has no
disc emission at all, so its "narrow" component is not meaningful - such lines are used only for the outflow.

In [ ]:
# optional: keep only spaxels where sigma_broad >= 1.3 x sigma_narrow
sigmas_two = btb.dispersion_medians(line_maps, min_ratio=1.3)
fig = bplt.plot_dispersion_histograms(line_maps, sigmas_two, min_ratio=1.3)

## Step 2 - velocity models

A number **fixes** a parameter; a tuple `(lo, hi)` makes it **free**. Here we fix the geometry and fit
the rest with fast least squares (use `method="emcee"` for a full analysis).

In [ ]:
GEOMETRY = dict(inc_disc=45, pa_disc=200, inc_out=50, pa_out=110)
FREE = dict(v_disc=(50, 800), R_disc=(20, 2000), v_out=(100, 1500), R_out=(50, 3000),
            theta_in=(0, 50), theta_out=(30, 110))

disc_fit = btb.fit_kinematics(line_maps["FeII"], "disc", {**GEOMETRY, **FREE}, method="least_squares")
out_fit  = btb.fit_kinematics(line_maps["NeV"], "outflow", {**GEOMETRY, **FREE}, method="least_squares")

In [ ]:
comb_fit = btb.fit_kinematics(line_maps["NeIII"], "combined", {**GEOMETRY, **FREE}, method="least_squares")
fig = bplt.plot_kinematic_fit(comb_fit, line_maps["NeIII"]);  proj.savefig(fig, "kinematic_fit_NeIII")

## Step 3 - break the blend

Every line gets the disc and outflow velocity fields (here from the fits above) and its own two dispersions.
Only the amplitudes are fitted in each spaxel.

In [ ]:
# .params holds every parameter at its final value (fixed or fitted), so any geometry
# you left free above is taken from the fit automatically
BASE = {**disc_fit.params, **out_fit.params}
DECOMPOSE = ["FeII", "ArII", "NeII", "SIII", "ArIII", "SIV", "NeIII", "OIV"]

params  = btb.line_params({n: line_maps[n] for n in DECOMPOSE}, BASE)
decomps = btb.decompose_lines(line_maps, params, sigmas, names=DECOMPOSE)
fig = bplt.plot_decomposition_spaxels(decomps["NeIII"], line_maps["NeIII"], n=6)
proj.savefig(fig, "decomposition_spaxels_NeIII")

In [ ]:
fig = bplt.plot_fraction_maps(decomps);               proj.savefig(fig, "disc_fraction_maps")
fig = bplt.plot_truth_comparison(decomps, truth);      proj.savefig(fig, "truth_comparison")   # recovered - true

for n, d in decomps.items():
    print(f"{d.label:>9s}: median |f_disc - truth| = {np.nanmedian(np.abs(d.f_disc - truth['f_disc'][n])):.3f}")

## Step 4 - stack

In [ ]:
stack = btb.stack_fraction_maps(decomps)
FRACTIONS = btb.fraction_summary(stack, sf_max=0.2, agn_min=0.8)    # share of the map dominated by each component
fig = bplt.plot_stacked_map(stack, title="Mock galaxy", scalebar_pc=300)
proj.savefig(fig, "stacked_outflow_fraction")
proj.export_stacked_map(stack)

The exported file `results/MockGalaxy/stacked_fraction_map.fits` has three extensions: the AGN outflow
fraction (primary), the disc fraction (`F_DISC`) and **`N_LINES`** - the number of emission lines that
contributed to each pixel of the stacked map (only pixels with at least `min_lines`, default 2, have a
fraction). It shows how well constrained each part of the map is. The same arrays are in the `.npz` file.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5.5), constrained_layout=True)
bplt.style_axes(ax)
im = ax.imshow(np.where(stack.n_lines > 0, stack.n_lines, np.nan), extent=stack.extent, cmap="viridis")
bplt.style_colorbar(plt.colorbar(im, ax=ax), "Number of lines")
ax.set_xlabel(r"$\Delta x$ [pc]"); ax.set_ylabel(r"$\Delta y$ [pc]")
proj.savefig(fig, "n_lines_map")

## Step 5 - AGN and star-forming spectra

In [ ]:
spectra = btb.extract_all(cubes, stack)
fig = bplt.plot_extraction_regions(spectra["ch3-long"], cubes["ch3-long"]);  proj.savefig(fig, "extraction_regions")
w_agn, f_agn, _ = btb.stitch_spectra(spectra, "agn", verbose=False)
w_sf,  f_sf,  _ = btb.stitch_spectra(spectra, "sf",  verbose=False)
fig = bplt.plot_component_spectra(w_agn, f_agn, w_sf, f_sf);  proj.savefig(fig, "component_spectra")

## Things to try

1. **Dispersions.** Repeat the decomposition with different fixed dispersions (for example swapping them, or using
   100 and 150 km s$^{-1}$) and compare the disc-fraction maps with the truth.
2. **Line choice and weighting.** Stack a subset of lines (e.g. `names=["NeII", "NeIII"]`) or use `scheme="equal"`,
   and compare the results with `btb.compare_stacks`.
3. **Outflow geometry.** Let `inc_out` vary in the outflow fit (e.g. `inc_out=(30, 70)`) and see how well it is
   constrained by these data.